# Base-model sampling — live from `cpsc_ckpt.pt`

The §1 opener beat, made tangible: *"about half of you trained this exact architecture.
Here is everything it learned to do: predict the next character."*

**Stage rules (rehearsed, not improvised):**

- Run **Setup** and confirm the param count *before* the talk starts.
- The 🎤 cells are the only ones touched live. Re-running is seconds on CPU.
- If anything misbehaves → scroll straight to **🛟 Fallback** (pre-generated output).
- No training happens here — checkpoint inference only (per the scope amendment).

**Requires:** `torch` + the checkpoint at `gpt_run_outputs/cpsc_ckpt.pt`
(model weights + vocab + config, saved by `gpt_train_instrumented.py`).

In [ ]:
# setup: load checkpoint (CPU) and rebuild the exact class model — no training
from pathlib import Path
import torch

from gpt_train_instrumented import GPTLanguageModel   # the class's model, verbatim

CKPT = Path("gpt_run_outputs/cpsc_ckpt.pt")           # <- adjust if your tag differs

try:
    ckpt = torch.load(CKPT, map_location="cpu")
except Exception:                                     # torch-version weights_only quirks
    ckpt = torch.load(CKPT, map_location="cpu", weights_only=False)

cfg, stoi, itos = ckpt["config"], ckpt["stoi"], ckpt["itos"]
model = GPTLanguageModel(cfg["vocab_size"], cfg["n_embd"], cfg["n_head"],
                         cfg["n_layer"], cfg["block_size"], cfg["dropout"])
model.load_state_dict(ckpt["model_state"])
model.eval()

n_params = sum(p.numel() for p in model.parameters())
print(f"torch {torch.__version__} | loaded {CKPT.name}")
print(f"vocab {cfg['vocab_size']} chars | block {cfg['block_size']} | "
      f"{n_params:,} params ({n_params/1e6:.2f}M) — ready on CPU")

In [ ]:
# helper: encode a prompt safely, sample, decode
def sample(prompt="", n_tokens=300, temperature=1.0, top_k=None, seed=None):
    """seed=None -> fresh text every run ("it's a distribution, not a lookup");
    seed=1337  -> the same text every run (safe for a rehearsed beat)."""
    if seed is not None:
        torch.manual_seed(seed)
    ids = [stoi[c] for c in prompt if c in stoi]
    dropped = sorted({c for c in prompt if c not in stoi})
    if dropped:
        print(f"(dropped chars the model has never seen: {dropped})")
    idx = torch.tensor([ids or [0]], dtype=torch.long)   # [0] = class script's empty start
    out = model.generate(idx, max_new_tokens=n_tokens,
                         temperature=temperature, top_k=top_k)
    text = "".join(itos[i] for i in out[0].tolist())
    print(text)
    return text

print("sample(prompt, n_tokens, temperature, top_k, seed) — ready")

---
## 🎤 Live 1 — the base-model beat

Talk track: it has learned the *shape* of an incident report — spelling, capitalization,
clause structure — from nothing but next-char prediction. And it's still semi-gibberish:
**that's what "base model" means.** Take an audience prompt if the room offers one.

In [ ]:
# 🎤 LIVE — fresh sample every run (add seed=1337 for a repeatable one)
_ = sample(prompt="THE CONSUMER REPORTED THAT THE ",
           n_tokens=300, temperature=0.8, top_k=40)

## 🎤 Live 2 — the temperature knob (optional)

Same prompt, three settings — the exact three from the class script.

In [ ]:
# 🎤 LIVE (optional) — same prompt, the knobs from gpt_train.py
for T, k in [(0.6, 20), (1.0, None), (1.3, 10)]:
    print(f"\n=== temperature={T}, top_k={k} ===")
    _ = sample("THE CONSUMER REPORTED THAT THE ", n_tokens=150,
               temperature=T, top_k=k, seed=1337)

## 🎤 Live 3 — what the model *actually* outputs (optional)

Not text: a **probability distribution over the next character**. Change the prompt,
watch the distribution move. (Slide version: `figures/gpt_nextchar_bars_cpsc.png`.)

In [ ]:
# 🎤 LIVE (optional) — P(next char | prompt)
import numpy as np
import matplotlib.pyplot as plt

prompt = "THE CONSUMER REPORTED THAT TH"        # <- edit live

show = lambda ch: {" ": "␣", "\n": "\\n", "\t": "\\t"}.get(ch, ch)
ids = [stoi[c] for c in prompt if c in stoi] or [0]
with torch.no_grad():
    logits, _ = model(torch.tensor([ids[-cfg["block_size"]:]]))
p = torch.softmax(logits[0, -1], dim=-1).numpy()

top = np.argsort(p)[::-1][:15]
y = np.arange(len(top))[::-1]
fig, ax = plt.subplots(figsize=(8, 5.5))
ax.barh(y, p[top], color=["#D55E00"] + ["#0072B2"] * (len(top) - 1))
ax.set_yticks(y, [show(itos[int(i)]) for i in top], family="monospace", fontsize=15)
ax.set_xlabel("P(next character)", fontsize=14)
ax.set_title(f'after “…{prompt[-20:]}”', fontsize=15, family="monospace")
ax.spines[["top", "right"]].set_visible(False)
ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()

---
## 🛟 Fallback — pre-generated output from the same checkpoint

If anything misbehaves live, show this instead (captured by the offline run).

In [ ]:
print(Path("gpt_run_outputs/cpsc_samples.txt").read_text(encoding="utf-8"))

> **TODO (zero-dependency fallback):** paste your 2–3 favorite generated snippets
> into *this* markdown cell — survives even a dead kernel.

---
**Env:** CPU-only inference; versions print in Setup. **Reproducibility:** everything here
traces to the one instrumented offline run (`gpt_train_instrumented.py`, seed 1337).